In [6]:
import numpy as np
import tensorflow as tf
import cv2

# ==================================================
# 1. LOAD MODEL
# ==================================================
model = tf.keras.models.load_model("cnn_model.h5")

conv1_w, conv1_b = model.layers[0].get_weights()

print("Model loaded.")
print("conv1_w shape:", conv1_w.shape)  # (3,3,1,8)
print("conv1_b shape:", conv1_b.shape)

# ==================================================
# 2. CONVERT TO Q8.8
# ==================================================
SCALE = 256

conv1_w_q = np.round(conv1_w * SCALE).astype(np.int16)
conv1_b_q = np.round(conv1_b * SCALE).astype(np.int16)

print("Q8.8 conversion done.")

# ==================================================
# 3. LOAD TEST IMAGE
# ==================================================
img = cv2.imread("050000_96.png", cv2.IMREAD_GRAYSCALE)
img = cv2.resize(img, (32, 32))
img = img.astype(np.float32) / 255.0

# add channel dimension (VERY IMPORTANT)
img = np.expand_dims(img, axis=-1)

print("Image shape:", img.shape)

# convert image to Q8.8
img_q = np.round(img * SCALE).astype(np.int16)

# ==================================================
# 4. FIXED-POINT CONV1 (LIKE VERILOG)
# ==================================================
H, W, C = img_q.shape
OUT = H - 2  # 30
FILTERS = 8

conv1_out = np.zeros((OUT, OUT, FILTERS), dtype=np.int16)

for f in range(FILTERS):
    for i in range(OUT):
        for j in range(OUT):
            acc = 0

            for ki in range(3):
                for kj in range(3):
                    pixel = int(img_q[i+ki, j+kj, 0])
                    weight = int(conv1_w_q[ki, kj, 0, f])
                    acc += pixel * weight  # 32-bit in python

            acc = acc >> 8          # SHIFT Q8.8
            acc += int(conv1_b_q[f])

            # clamp to int16 range
            if acc > 32767:
                acc = 32767
            elif acc < -32768:
                acc = -32768

            conv1_out[i, j, f] = acc

print("Conv1 fixed done.")

# ==================================================
# 5. PRINT GOLDEN REFERENCE
# ==================================================
print("\nGolden reference values:")
print("conv1_out[0,0,0] =", conv1_out[0,0,0])
print("conv1_out[0,1,0] =", conv1_out[0,1,0])
print("conv1_out[1,0,0] =", conv1_out[1,0,0])

# ==================================================
# 6. EXPORT FILES FOR VERILOG
# ==================================================

# -------- export image --------
with open("img.mem", "w") as f:
    for i in range(32):
        for j in range(32):
            val = int(img_q[i,j,0]) & 0xFFFF
            f.write(f"{val:04x}\n")

print("img.mem exported")

# -------- export conv1 weight (FILTER MAJOR ORDER) --------
with open("conv1_w.mem", "w") as f:
    for f_id in range(8):          # filter ngoài cùng
        for kr in range(3):
            for kc in range(3):
                val = conv1_w_q[kr, kc, 0, f_id]
                val = int(val) & 0xFFFF
                f.write(f"{val:04x}\n")

print("conv1_w.mem exported (filter-major order)")

# -------- export conv1 bias --------
with open("conv1_b.mem", "w") as f:
    for val in conv1_b_q:
        val = int(val) & 0xFFFF
        f.write(f"{val:04x}\n")

print("conv1_b.mem exported")

print("\nAll files ready for Verilog.")

Model loaded.
conv1_w shape: (3, 3, 1, 8)
conv1_b shape: (8,)
Q8.8 conversion done.
Image shape: (32, 32, 1)
Conv1 fixed done.

Golden reference values:
conv1_out[0,0,0] = 33
conv1_out[0,1,0] = 30
conv1_out[1,0,0] = 31
img.mem exported
conv1_w.mem exported (filter-major order)
conv1_b.mem exported

All files ready for Verilog.
